# Stage 2: Frozen TimesFM representation with text fusion

This notebook uses exactly one frozen pretrained price encoder, TimesFM, as specified in Section 8.1 of `latex/main.tex`. All saved feature columns are read from the artifacts created by `data_analysis.ipynb` and classified by forecast-time availability. TimesFM 2.5's hidden-token interface is univariate, so the frozen representation is extracted from the saved `ret_20` history only; the notebook does not falsely claim that its external XReg covariates alter the transformer hidden tokens.

For every purged expanding fold, text coders, covariate preprocessing, and residual fusion networks are fitted using only that fold's training rows while TimesFM remains frozen. Validation predictions are genuinely out of sample. Bayesian tuning uses only those validation blocks; afterward the selected architecture is refitted on all training data, evaluated separately on 2022 and 2023 test targets, and used to create organizer-format submissions for both years.


In [1]:
# %pip install optuna

In [ ]:
from pathlib import Path
import json
import numpy as np
import polars as pl
import torch
import importlib
import inspect
from latent_fusion import generate_timesfm_price_latents, run_walk_forward_fusion


DATA_DIR = Path('data')
ARTIFACT_DIR = DATA_DIR / 'model_artifacts' / 'pca_embeddings'
BASELINE_DIR = ARTIFACT_DIR / 'baselines'
OUTPUT_DIR = ARTIFACT_DIR / 'stage2_pretrained_forecasts' / 'timesfm_covariate_residual_fusion'
PRICE_CACHE_DIR = ARTIFACT_DIR / 'stage2_pretrained_forecasts' / 'timesfm_simple_concatenation' / 'price_latents'
PREPARED_TRAIN_PATH = ARTIFACT_DIR / 'train_features_without_embeddings.parquet'
PREPARED_TEST_PATH = ARTIFACT_DIR / 'test_features_without_embeddings.parquet'
TRAIN_TARGET_PATH = ARTIFACT_DIR / 'train_target.parquet'
TEST_TARGET_PATH = ARTIFACT_DIR / 'test_target.parquet'
TRAIN_LINK_PATH = ARTIFACT_DIR / 'train_text_links.parquet'
TEST_LINK_PATH = ARTIFACT_DIR / 'test_text_links.parquet'
FOLD_PATH = ARTIFACT_DIR / 'walk_forward_assignments.parquet'

PRICE_ENCODER_NAME = 'timesfm'
PRICE_ENCODER_MODEL_ID = 'google/timesfm-2.5-200m-pytorch'
HORIZON, LOOKBACK, MIN_CONTEXT = 20, 512, 64
TIMESFM_INPUT_COLUMN = 'ret_20'
REQUESTED_TEXT_FAMILIES = ('bert',)  # add 'gemini', 'nvda', 'lgai', 'qwen' as needed
TEXT_LATENT_DIM, FUSION_HIDDEN_DIM = 128, 256
MARKET_DEPTH, FUSION_DEPTH, RESIDUAL_EXPANSION, FUSION_DROPOUT = 2, 2, 2, 0.10
TEXT_CODER_EPOCHS, FUSION_EPOCHS = 3, 10
PRICE_BATCH_SIZE, TEXT_BATCH_SIZE, FUSION_BATCH_SIZE = 16, 1024, 512
TUNE_HYPERPARAMETERS, OPTUNA_TRIALS = True, 12
SUBMISSION_YEARS, EXPECTED_SUBMISSION_ROWS = (2022, 2023), 52_000
RANDOM_STATE = 42
RUN_PRICE_LATENT_EXTRACTION = True
RUN_MODEL_TRAINING = True
FORCE_REFRESH_PRICE_LATENTS = False
FORCE_REFRESH_MODELS = False
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: mps


In [3]:
required_paths = (
    PREPARED_TRAIN_PATH, PREPARED_TEST_PATH, TRAIN_TARGET_PATH, TEST_TARGET_PATH,
    TRAIN_LINK_PATH, TEST_LINK_PATH, FOLD_PATH,
)
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

train_features = pl.read_parquet(PREPARED_TRAIN_PATH).sort(['ticker', 'date'])
test_features = pl.read_parquet(PREPARED_TEST_PATH).sort(['ticker', 'date'])
if train_features.columns != test_features.columns:
    raise ValueError('Train and test engineered-feature schemas differ.')
ID_COLUMNS = ('row_id', 'date', 'ticker')
TEXT_AVAILABILITY_PREFIXES = (
    'macro_', 'sector_category_', 'target_company_', 'related_company_', 'filing_',
)
PAST_TEXT_AVAILABILITY_COVARIATES = tuple(
    column for column in train_features.columns
    if column.startswith(TEXT_AVAILABILITY_PREFIXES) or column.startswith('has_')
    or column in ('text_count_total', 'unique_text_id_count')
)
KNOWN_FUTURE_PREFIXES = (
    'calendar_', 'trading_day_', 'trading_days_', 'days_since_start',
    'is_month_', 'is_first_', 'is_last_', 'is_quarter_end',
    'trading_week_fourier_', 'month_of_year_fourier_', 'trading_month_fourier_',
)
KNOWN_FUTURE_COVARIATES = tuple(
    column for column in train_features.columns if column.startswith(KNOWN_FUTURE_PREFIXES)
)
PAST_MARKET_COVARIATES = tuple(
    column for column, dtype in train_features.schema.items()
    if dtype.is_numeric() and column not in ID_COLUMNS
    and column not in KNOWN_FUTURE_COVARIATES
    and column not in PAST_TEXT_AVAILABILITY_COVARIATES
)
if TIMESFM_INPUT_COLUMN not in PAST_MARKET_COVARIATES:
    raise ValueError(f'{TIMESFM_INPUT_COLUMN} is absent from historical market features.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'feature_groups.json').write_text(json.dumps({
    'timesfm_hidden_input': TIMESFM_INPUT_COLUMN,
    'past_market_covariates': list(PAST_MARKET_COVARIATES),
    'known_future_covariates': list(KNOWN_FUTURE_COVARIATES),
    'past_text_availability_covariates': list(PAST_TEXT_AVAILABILITY_COVARIATES),
    'timesfm_hidden_covariate_support': 'univariate_target_only',
}, indent=2))
MODEL_COVARIATES = (
    *PAST_MARKET_COVARIATES, *KNOWN_FUTURE_COVARIATES,
    *PAST_TEXT_AVAILABILITY_COVARIATES,
)
if len(MODEL_COVARIATES) != len(set(MODEL_COVARIATES)):
    raise ValueError('Engineered covariate groups overlap.')
train_targets = pl.read_parquet(TRAIN_TARGET_PATH).select(
    ['row_id', 'date', 'ticker', 'fwd_log_return_20', 'target_up']
)
test_targets = pl.read_parquet(TEST_TARGET_PATH).select(
    ['row_id', 'date', 'ticker', 'fwd_log_return_20', 'target_up']
)
train_links = pl.read_parquet(TRAIN_LINK_PATH)
test_links = pl.read_parquet(TEST_LINK_PATH)
fold_assignments = pl.read_parquet(FOLD_PATH)
train_origins = train_targets.filter(pl.col('target_up').is_not_null()).select(
    ['row_id', 'date', 'ticker']
)
test_origins = test_features.select(['row_id', 'date', 'ticker'])
print('prepared features:', train_features.shape, test_features.shape)
print(f'past market={len(PAST_MARKET_COVARIATES)}, known future={len(KNOWN_FUTURE_COVARIATES)}, text availability={len(PAST_TEXT_AVAILABILITY_COVARIATES)}')
print('PAST_MARKET_COVARIATES =', list(PAST_MARKET_COVARIATES))
print('KNOWN_FUTURE_COVARIATES =', list(KNOWN_FUTURE_COVARIATES))
print('walk-forward folds:', fold_assignments['fold'].unique().sort().to_list())


prepared features: (104400, 109) (128400, 109)
past market=66, known future=28, text availability=12
PAST_MARKET_COVARIATES = ['ret_1', 'log_hl_range', 'log_close_open', 'ret_5', 'ret_mean_5', 'ret_std_5', 'ma_gap_5', 'drawdown_5', 'ret_10', 'ret_mean_10', 'ret_std_10', 'ma_gap_10', 'drawdown_10', 'ret_20', 'ret_mean_20', 'ret_std_20', 'ma_gap_20', 'drawdown_20', 'ret_40', 'ret_mean_40', 'ret_std_40', 'ma_gap_40', 'drawdown_40', 'ret_60', 'ret_mean_60', 'ret_std_60', 'ma_gap_60', 'drawdown_60', 'momentum_accel_5', 'ret_skew_20', 'ret_kurt_20', 'ret_skew_60', 'ret_kurt_60', 'log_volume', 'volume_change_1', 'log_volume_mean_5', 'log_volume_std_5', 'log_volume_gap_5', 'volume_z_5', 'volume_ma_gap_5', 'log_volume_mean_10', 'log_volume_std_10', 'log_volume_gap_10', 'volume_z_10', 'volume_ma_gap_10', 'log_volume_mean_20', 'log_volume_std_20', 'log_volume_gap_20', 'volume_z_20', 'volume_ma_gap_20', 'log_volume_mean_40', 'log_volume_std_40', 'log_volume_gap_40', 'volume_z_40', 'volume_ma_gap_4

## 1. Extract one frozen price representation

TimesFM receives the saved backward-looking 20-day return sequence. Its pretrained parameters are frozen. The final transformer patch tokens corresponding to valid context values are mean-pooled and cached as $h^P_{i,t}$. These caches are label independent and can be reused by every walk-forward fold.


In [4]:
price_cache_dir = PRICE_CACHE_DIR  # label-independent cache reused by the new fusion model
train_price_latents = generate_timesfm_price_latents(
    split='train', prepared_features=train_features, origins=train_origins,
    cache_path=price_cache_dir / 'train_timesfm_pooled_hidden.parquet',
    model_id=PRICE_ENCODER_MODEL_ID, device=DEVICE, horizon=HORIZON,
    lookback=LOOKBACK, min_context=MIN_CONTEXT,
    batch_size=PRICE_BATCH_SIZE, run_extraction=RUN_PRICE_LATENT_EXTRACTION,
    force_refresh=FORCE_REFRESH_PRICE_LATENTS,
)
test_price_latents = generate_timesfm_price_latents(
    split='test', prepared_features=test_features, origins=test_origins,
    cache_path=price_cache_dir / 'test_timesfm_pooled_hidden.parquet',
    model_id=PRICE_ENCODER_MODEL_ID, device=DEVICE, horizon=HORIZON,
    lookback=LOOKBACK, min_context=MIN_CONTEXT,
    batch_size=PRICE_BATCH_SIZE, run_extraction=RUN_PRICE_LATENT_EXTRACTION,
    force_refresh=FORCE_REFRESH_PRICE_LATENTS,
)
print('price latent caches:', train_price_latents.shape, test_price_latents.shape)


price latent caches: (94100, 1283) (120100, 1283)


## 2. Fold-local text coders, residual market MLP, and fusion

For fold $k$, every family-specific text coder is trained only from that fold's training text IDs and labels. It projects each original article embedding before articles are averaged within the same stock-date and family. The frozen TimesFM latent is concatenated with all engineered market, calendar, and text-availability covariates. Missing covariates are median-imputed and standardized using that fold's training rows only. A residual MLP encodes this combined numeric branch, which is then concatenated with the family-specific text branch and passed through a second residual fusion head.

Optuna's TPE sampler selects the shared residual architecture using mean balanced accuracy across the five expanding validation folds; test rows are never used for tuning. The chosen model is then refitted on all labeled training rows. Metrics are saved separately for prediction years 2022 and 2023, and each model writes one 52,000-row submission CSV containing both years. Install the optional tuner in the active kernel with `%pip install optuna` before running when `TUNE_HYPERPARAMETERS=True`.


In [5]:
results = run_walk_forward_fusion(
    data_dir=DATA_DIR, output_dir=OUTPUT_DIR, baseline_dir=BASELINE_DIR,
    train_price_latents=train_price_latents, test_price_latents=test_price_latents,
    train_features=train_features, test_features=test_features,
    train_links=train_links, test_links=test_links,
    train_targets=train_targets, test_targets=test_targets,
    fold_assignments=fold_assignments, requested_families=REQUESTED_TEXT_FAMILIES,
    covariate_columns=MODEL_COVARIATES,
    device=DEVICE, latent_dim=TEXT_LATENT_DIM, text_epochs=TEXT_CODER_EPOCHS,
    fusion_epochs=FUSION_EPOCHS, text_batch_size=TEXT_BATCH_SIZE,
    fusion_batch_size=FUSION_BATCH_SIZE, fusion_hidden_dim=FUSION_HIDDEN_DIM,
    market_depth=MARKET_DEPTH, fusion_depth=FUSION_DEPTH,
    residual_expansion=RESIDUAL_EXPANSION,
    fusion_dropout=FUSION_DROPOUT, tuning_trials=OPTUNA_TRIALS,
    tune_hyperparameters=TUNE_HYPERPARAMETERS,
    forecast_horizon_weekdays=HORIZON, submission_years=SUBMISSION_YEARS,
    expected_submission_rows=EXPECTED_SUBMISSION_ROWS,
    raw_test_path=DATA_DIR / 'test.parquet',
    seed=RANDOM_STATE, run_training=RUN_MODEL_TRAINING,
    force_refresh=FORCE_REFRESH_MODELS,
)
display(results['fold_metrics'])
display(results['aggregate'])
display(results['comparison_aggregate'])
display(results['final_metrics'])
display(results['submission_manifest'])


/Users/hoanguyen/research/KDD 2026 - Time series challenge/latent_fusion.py:1138: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(seed=seed, multivariate=True),


feature_set,model,config_id,params_json,decision_threshold,fold,train_rows_used,validation_rows,accuracy,balanced_accuracy,roc_auc,brier_score
str,str,str,str,f64,i64,i64,i64,f64,f64,f64,f64
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""","""timesfm_covariates_plus_bert__…","""{""dropout"": 0.2848747353910473…",0.65,1,25600,14900,0.590268,0.553104,0.610177,0.34563
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""","""timesfm_covariates_plus_bert__…","""{""dropout"": 0.2848747353910473…",0.65,2,40500,14900,0.579799,0.548669,0.560948,0.374262
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""","""timesfm_covariates_plus_bert__…","""{""dropout"": 0.2848747353910473…",0.65,3,55400,14900,0.521275,0.501969,0.489817,0.322608
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""","""timesfm_covariates_plus_bert__…","""{""dropout"": 0.2848747353910473…",0.65,4,70300,14900,0.507181,0.49569,0.500287,0.382256
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""","""timesfm_covariates_plus_bert__…","""{""dropout"": 0.2848747353910473…",0.65,5,85200,15200,0.512105,0.511151,0.523159,0.38082


feature_set,model,config_id,params_json,completed_folds,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,mean_roc_auc,std_roc_auc,mean_brier_score,decision_threshold
str,str,str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""","""timesfm_covariates_plus_bert__…","""{""dropout"": 0.2848747353910473…",5,0.542126,0.039668,0.522117,0.026878,0.536878,0.049229,0.361115,0.65


feature_set,model,config_id,params_json,completed_folds,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,mean_roc_auc,std_roc_auc,mean_brier_score,decision_threshold
str,str,str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""article_mean_pca_embeddings""","""logistic_l2""","""logistic_l2__03""","""{""C"": 10.0}""",5,0.483475,0.034672,0.526041,0.018227,0.512895,0.048122,0.319034,0.65
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""","""timesfm_covariates_plus_bert__…","""{""dropout"": 0.2848747353910473…",5,0.542126,0.039668,0.522117,0.026878,0.536878,0.049229,0.361115,0.65
"""article_mean_pca_embeddings""","""reversal_20""","""reversal_20__00""","""{}""",5,0.502911,0.020459,0.519412,0.01061,0.519412,0.01061,0.497089,0.5
"""article_mean_pca_embeddings""","""catboost""","""catboost__00""","""{""bagging_temperature"": 1.0, ""…",5,0.543484,0.041124,0.509731,0.033285,0.516813,0.060457,0.277333,0.45
"""article_mean_pca_embeddings""","""xgboost""","""xgboost__03""","""{""colsample_bytree"": 0.8, ""gam…",5,0.552871,0.062747,0.505198,0.024765,0.508945,0.063898,0.288579,0.35
"""article_mean_pca_embeddings""","""lightgbm""","""lightgbm__03""","""{""colsample_bytree"": 0.8, ""lea…",5,0.552377,0.062024,0.503282,0.024996,0.513142,0.06105,0.293511,0.35
"""article_mean_pca_embeddings""","""always_up""","""always_up__00""","""{}""",5,0.559257,0.078075,0.5,0.0,0.5,0.0,0.440743,0.5
"""article_mean_pca_embeddings""","""momentum_20""","""momentum_20__00""","""{}""",5,0.496955,0.020435,0.480526,0.010718,0.480526,0.010718,0.503045,0.5


feature_set,model,test_year,total_prediction_rows,scored_rows,coverage,decision_threshold,hit_rate,accuracy,balanced_accuracy,roc_auc,brier_score
str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""",2022,26000,26000,1.0,0.65,0.861769,0.861769,0.859212,0.939973,0.099586
"""timesfm_covariates_original_te…","""timesfm_covariates_plus_bert""",2023,26000,24000,0.923077,0.65,0.519708,0.519708,0.532165,0.548628,0.372674


model,path,rows,years,return_scale,decision_threshold
str,str,i64,str,f64,f64
"""timesfm_covariates_plus_bert""","""data/model_artifacts/pca_embed…",52000,"""2022,2023""",0.051065,0.65


## Saved outputs

`fold_results/fold_<k>/` contains fold-local text coders, covariate scalers, residual-fusion checkpoints, histories, and OOS metrics. `optuna_trials.csv`, `best_hyperparameters.json`, fold/aggregate CSV summaries, decision thresholds, OOF Parquet predictions, and baseline comparisons are saved at the root. `final_refit/` contains all-training refits. `final_test_predictions.parquet` and `final_test_metrics_by_year.csv` hold test outputs; `submissions/` contains one organizer-format 2022--2023 CSV per model and `submission_manifest.csv` indexes them.
